# 03. NV3: Mô phỏng và Đánh giá chính sách phí nhàn rỗi (Idle Fee Policy)

## Mục tiêu nghiên cứu (NV3)
Đánh giá tác động kinh tế và hành vi của các kịch bản áp dụng phí nhàn rỗi (Overstay/Idle Fee):
1. **Kịch bản cơ sở (Baseline):** Không thu phí nhàn rỗi.
2. **Kịch bản 1 (Phí bậc thang đơn giản):** Miễn phí 30 phút đầu sau khi sạc xong (Grace period = 30m), sau đó tính phí \$0.20/phút.
3. **Kịch bản 2 (Phí lũy tiến nghiêm ngặt):** Miễn phí 15 phút đầu, 15–60 phút tính \$0.40/phút, sau 60 phút tính \$1.00/phút.
4. Ước tính doanh thu bổ sung cho đơn vị vận hành trạm và mức độ giảm thiểu thời gian chiếm giữ lãng phí.


In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 150
OUTPUT_DIR = "outputs_nv3_v6"
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(42)

### 1. NẠP VÀ AUDIT DỮ LIỆU (TẠO BẢNG AUDIT)


In [5]:
print(">>> CELL 1: NẠP VÀ AUDIT DỮ LIỆU NGHIÊM NGẶT")

FILE_PATH = "../data/processed/sessions_features.parquet"
try:
    df = pd.read_parquet(FILE_PATH) if FILE_PATH.endswith(".parquet") else pd.read_csv(FILE_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Không tìm thấy file tại {FILE_PATH}")

rename_dict = {
    'charging_time_hours': 'T_charging', 
    'idle_time_hours': 'T_idle', 
    'total_duration_hours': 'T_total', 
    'Energy (kWh)': 'Energy', 
    'idle_ratio': 'Idle_Ratio'
}
df = df.rename(columns=rename_dict)

required_cols = ['Start Date', 'T_total', 'T_charging', 'T_idle', 'Idle_Ratio', 'Energy', 'User ID', 'Driver Postal Code']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Thiếu các cột bắt buộc: {missing_cols}")

N_INITIAL = len(df)
audit_log = []

# 1. Kiểm tra ngày tháng
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce")
n_date_err = df["Start Date"].isna().sum()
if n_date_err > 0:
    audit_log.append({
        "Kiểm tra": "Start Date không parse được", 
        "Số dòng": n_date_err, 
        "Tỷ lệ (%)": round(100 * n_date_err / N_INITIAL, 4), 
        "Xử lý": "Loại bỏ khỏi phân tích"
    })

# 2. Ép kiểu số và xử lý vô hạn
numeric_cols = ['T_total', 'T_charging', 'T_idle', 'Idle_Ratio', 'Energy']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[~np.isfinite(df[col]), col] = np.nan

# 3. Đối chiếu T_idle
df["T_idle_check"] = df["T_total"] - df["T_charging"]
checkable_idle = df[["T_idle", "T_total", "T_charging"]].notna().all(axis=1)
mismatch_idle = checkable_idle & ~np.isclose(df["T_idle"], df["T_idle_check"], atol=1e-4, rtol=0)
audit_log.append({
    "Kiểm tra": "T_idle lệch (T_total - T_charging)", 
    "Số dòng": mismatch_idle.sum(),
    "Tỷ lệ (%)": round(100 * mismatch_idle.sum() / max(checkable_idle.sum(), 1), 4),
    "Xử lý": "Giữ nguyên T_idle gốc, ghi nhận cảnh báo"
})

# 4. Đối chiếu Idle_Ratio
df["Idle_Ratio_check"] = np.where(df["T_total"] > 0, df["T_idle"] / df["T_total"], np.nan)
checkable_ratio = df[["Idle_Ratio", "Idle_Ratio_check"]].notna().all(axis=1)
mismatch_ratio = checkable_ratio & ~np.isclose(df["Idle_Ratio"], df["Idle_Ratio_check"], atol=1e-4, rtol=0)
audit_log.append({
    "Kiểm tra": "Idle_Ratio lệch (T_idle/T_total)", 
    "Số dòng": mismatch_ratio.sum(),
    "Tỷ lệ (%)": round(100 * mismatch_ratio.sum() / max(checkable_ratio.sum(), 1), 4),
    "Xử lý": "Dùng biến tái tính Idle_Ratio_check cho phân tích"
})

# 5. Kiểm tra giá trị âm và T_total <= 0
for col in ['T_idle', 'Energy', 'T_charging']:
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        audit_log.append({
            "Kiểm tra": f"{col} âm", 
            "Số dòng": n_neg,
            "Tỷ lệ (%)": round(100 * n_neg / N_INITIAL, 4), 
            "Xử lý": "Loại khỏi phân tích metric đó"
        })

n_nonpos_total = (df["T_total"] <= 0).sum()
if n_nonpos_total > 0:
    audit_log.append({
        "Kiểm tra": "T_total <= 0", 
        "Số dòng": n_nonpos_total,
        "Tỷ lệ (%)": round(100 * n_nonpos_total / N_INITIAL, 4), 
        "Xử lý": "Loại khỏi toàn bộ phân tích thời lượng và tỷ lệ"
    })

audit_df = pd.DataFrame(audit_log)
print("\n--- BẢNG AUDIT DỮ LIỆU (NV3) ---")
print(audit_df.to_markdown(index=False))
audit_df.to_csv(f"{OUTPUT_DIR}/audit_table.csv", index=False)

# Chuyển sang phút
df["T_idle_min"] = df["T_idle"] * 60

# Phân đoạn Pre/Post
POLICY_DATE = pd.Timestamp("2017-08-01")
df["Policy_Period"] = np.where(df["Start Date"] < POLICY_DATE, "Pre", "Post")
df = df.dropna(subset=["Start Date"])

>>> CELL 1: NẠP VÀ AUDIT DỮ LIỆU NGHIÊM NGẶT

--- BẢNG AUDIT DỮ LIỆU (NV3) ---
| Kiểm tra                           |   Số dòng |   Tỷ lệ (%) | Xử lý                                             |
|:-----------------------------------|----------:|------------:|:--------------------------------------------------|
| T_idle lệch (T_total - T_charging) |         0 |           0 | Giữ nguyên T_idle gốc, ghi nhận cảnh báo          |
| Idle_Ratio lệch (T_idle/T_total)   |         0 |           0 | Dùng biến tái tính Idle_Ratio_check cho phân tích |


### 2. PHÂN KHÚC NGƯỜI DÙNG & KIỂM TRA CẤU TRÚC


In [6]:
# ==============================================================================
# CELL 2: PHÂN KHÚC NGƯỜI DÙNG & KIỂM TRA CẤU TRÚC
# ==============================================================================
print("\n>>> CELL 2: PHÂN KHÚC NGƯỜI DÙNG VÀ KIỂM TRA CẤU TRÚC")

user_periods = (
    df.dropna(subset=["User ID"])
      .groupby("User ID")["Policy_Period"]
      .agg(lambda s: set(s))
)
n_pre_only = sum(v == {"Pre"} for v in user_periods)
n_post_only = sum(v == {"Post"} for v in user_periods)
n_both = sum(v == {"Pre", "Post"} for v in user_periods)

print(f" - User chỉ có mặt ở giai đoạn Pre: {n_pre_only:,}")
print(f" - User chỉ có mặt ở giai đoạn Post: {n_post_only:,}")
print(f" - User có mặt ở CẢ 2 giai đoạn (Paired): {n_both:,}")
print("   [Lưu ý] Phân tích chính so sánh các phiên sạc giữa hai giai đoạn Pre/Post,")
print("           không ghép cặp phiên sạc trước-sau của từng User.")
print("           Một số User xuất hiện ở cả hai giai đoạn; Cluster Bootstrap")
print("           lấy mẫu theo User để giữ các phiên thuộc cùng một cụm.")
print("           Kết quả vẫn cần được diễn giải trong bối cảnh khác biệt về thời gian")
print("           và các yếu tố gây nhiễu có thể có.")

# Phân khúc (Dùng cho mô tả)
pre_policy_df = df[df["Policy_Period"] == "Pre"]
user_freq_pre = pre_policy_df.groupby("User ID").size().reset_index(name="Pre_Session_Count")
df = df.merge(user_freq_pre, on="User ID", how="left")
df["Pre_Session_Count"] = df["Pre_Session_Count"].fillna(0)

df["Zip_Clean"] = df["Driver Postal Code"].astype(str).str.extract(r'(\d{5})')[0]
local_zips = ["94301", "94303", "94306", "94304", "94305"]

is_frequent = df["Pre_Session_Count"] >= 10
is_local = df["Zip_Clean"].isin(local_zips)

df["User_Segment"] = np.where(is_frequent & is_local, "Regular", "Casual")
df.loc[df["User ID"].isna() | df["Zip_Clean"].isna(), "User_Segment"] = "Unclassified"

user_strat = pd.crosstab(df["Policy_Period"], df["User_Segment"], normalize="index") * 100
print("\nTỷ lệ phân khúc (%):")
print(user_strat.round(2).to_markdown())
user_strat.to_csv(f"{OUTPUT_DIR}/3.3.2_user_strat.csv", index=False)



>>> CELL 2: PHÂN KHÚC NGƯỜI DÙNG VÀ KIỂM TRA CẤU TRÚC
 - User chỉ có mặt ở giai đoạn Pre: 5,813
 - User chỉ có mặt ở giai đoạn Post: 11,862
 - User có mặt ở CẢ 2 giai đoạn (Paired): 3,766
   [Lưu ý] Phân tích chính so sánh các phiên sạc giữa hai giai đoạn Pre/Post,
           không ghép cặp phiên sạc trước-sau của từng User.
           Một số User xuất hiện ở cả hai giai đoạn; Cluster Bootstrap
           lấy mẫu theo User để giữ các phiên thuộc cùng một cụm.
           Kết quả vẫn cần được diễn giải trong bối cảnh khác biệt về thời gian
           và các yếu tố gây nhiễu có thể có.

Tỷ lệ phân khúc (%):
| Policy_Period   |   Casual |   Regular |   Unclassified |
|:----------------|---------:|----------:|---------------:|
| Post            |    87.46 |       9.6 |           2.93 |
| Pre             |    67.83 |      28.5 |           3.67 |


## 3.Kiểm định Thống kê & Cluster Bootstrap

In [ ]:
# ==============================================================================
# CELL 3: KIỂM ĐỊNH THỐNG KÊ (MWU THAM KHẢO + CLUSTER BOOTSTRAP SUY LUẬN)
# ==============================================================================
print("\n>>> CELL 3: KIỂM ĐỊNH (MWU THAM KHẢO + CLUSTER BOOTSTRAP SUY LUẬN)")

def cliffs_delta(pre, post):
    pre, post = np.asarray(pre), np.asarray(post)
    if len(pre) == 0 or len(post) == 0: return np.nan
    u_stat, _ = stats.mannwhitneyu(pre, post, alternative="two-sided")
    return (2 * u_stat) / (len(pre) * len(post)) - 1

def interpret_delta(d):
    if not np.isfinite(d): return "Invalid"
    v = abs(d)
    if v < 0.147: return "Negligible"
    elif v < 0.330: return "Small"
    elif v < 0.474: return "Medium"
    else: return "Large"

def prepare_metric_data(data, metric):
    valid = data.dropna(subset=["User ID", "Policy_Period", metric]).copy()
    valid = valid[np.isfinite(valid[metric])]
    
    # Đồng bộ quy tắc loại T_total <= 0 với Audit
    valid = valid[valid["T_total"] > 0]
    
    if metric == "T_idle_min": valid = valid[valid[metric] >= 0]
    elif metric == "Energy": valid = valid[valid[metric] >= 0]
    elif metric == "T_charging": valid = valid[valid[metric] > 0]
    elif metric == "Idle_Ratio_check": valid = valid[(valid[metric] >= 0) & (valid[metric] <= 1)]
        
    return valid

def bootstrap_delta_ci(valid_data, metric, n_boot=2000):
    rng = np.random.default_rng(42)
    users = valid_data["User ID"].unique()
    if len(users) < 2: return (np.nan, np.nan)
    
    user_groups = {u: g for u, g in valid_data.groupby("User ID")}
    deltas = []
    
    for _ in range(n_boot):
        sampled_users = rng.choice(users, size=len(users), replace=True)
        boot_df = pd.concat([user_groups[u] for u in sampled_users], ignore_index=True)
        
        pre_vals = boot_df.loc[boot_df["Policy_Period"] == "Pre", metric].to_numpy()
        post_vals = boot_df.loc[boot_df["Policy_Period"] == "Post", metric].to_numpy()
        
        if len(pre_vals) > 0 and len(post_vals) > 0:
            deltas.append(cliffs_delta(pre_vals, post_vals))
    
    if len(deltas) < 0.95 * n_boot:
        print(f"   [CẢNH BÁO] {metric}: Chỉ {len(deltas)}/{n_boot} lần bootstrap thành công. CI không đáng tin.")
        return (np.nan, np.nan)
    return tuple(np.percentile(deltas, [2.5, 97.5]))

metrics = ["T_idle_min", "Idle_Ratio_check", "Energy", "T_charging"]
results = []
raw_pvals = []

for m in metrics:
    valid_df = prepare_metric_data(df, m)
    pre_vals = valid_df[valid_df["Policy_Period"] == "Pre"][m].to_numpy()
    post_vals = valid_df[valid_df["Policy_Period"] == "Post"][m].to_numpy()
    
    if len(pre_vals) == 0 or len(post_vals) == 0:
        print(f"   [BỎ QUA] {m}: Không đủ dữ liệu ở cả 2 giai đoạn.")
        continue
        
    delta = cliffs_delta(pre_vals, post_vals)
    _, pval = stats.mannwhitneyu(pre_vals, post_vals, alternative="two-sided")
    raw_pvals.append(pval)
    
    ci_lower, ci_upper = bootstrap_delta_ci(valid_df, m, n_boot=2000)
    
    results.append({
        "Metric": m, "N_Pre": len(pre_vals), "N_Post": len(post_vals),
        "Delta": round(delta, 4) if np.isfinite(delta) else np.nan,
        "CI 95% (Cluster)": f"[{ci_lower:.3f}, {ci_upper:.3f}]" if np.isfinite(ci_lower) else "N/A",
        "p-value (MWU)*": f"{pval:.2e}", "Mức độ (Cliff)": interpret_delta(delta)
    })

if not raw_pvals:
    raise ValueError("Không có metric nào đủ dữ liệu ở cả Pre và Post. Dừng phân tích.")

reject_holm, pvals_holm, _, _ = multipletests(raw_pvals, method='holm')
for i, r in enumerate(results):
    r["p-value (Holm)"] = f"{pvals_holm[i]:.2e}"
    r["Có ý nghĩa (Holm)"] = "Có" if reject_holm[i] else "Không"

res_df = pd.DataFrame(results)

print("\n(*) PHƯƠNG PHÁP SUY LUẬN:")
print("    - Mann-Whitney p-value (cột *): CHỈ THAM KHẢO vì vi phạm giả định độc lập (phiên lặp theo user).")
print("    - CI 95% (Cluster Bootstrap): Ước lượng độ bất định của Cliff's Delta,")
print("      có xét cấu trúc phiên sạc lặp theo User ID.")
print("      CI không tự kiểm soát nhiễu theo thời gian hoặc chứng minh quan hệ nhân quả.")
print("    - Cách diễn giải: Nếu CI 95% KHÔNG chứa giá trị 0, kết quả GỢI Ý có sự khác biệt")
print("      về thứ hạng tương đối giữa hai giai đoạn. Kết quả này không tự chứng minh")
print("      chính sách là nguyên nhân tạo ra khác biệt.")
print(res_df.to_markdown(index=False))
res_df.to_csv(f"{OUTPUT_DIR}/3.3.3_stats.csv", index=False)


>>> CELL 3: KIỂM ĐỊNH (MWU THAM KHẢO + CLUSTER BOOTSTRAP SUY LUẬN)


## 4: BUNCHING ANALYSIS & TRỰC QUAN HÓA

In [ ]:
print("\n>>> CELL 4: BUNCHING ANALYSIS")

valid_idle = df["T_idle_min"].notna() & np.isfinite(df["T_idle_min"]) & (df["T_idle_min"] >= 0)
pre_bunch = df[(df["Policy_Period"] == "Pre") & valid_idle]
post_bunch = df[(df["Policy_Period"] == "Post") & valid_idle]

def rate_pct(count, denominator):
    if denominator == 0: return np.nan
    return 100 * count / denominator

def count_range(data, low, high):
    return ((data["T_idle_min"] >= low) & (data["T_idle_min"] < high)).sum()

rate_pre_15_20 = rate_pct(count_range(pre_bunch, 15, 20), len(pre_bunch))
rate_pre_20_25 = rate_pct(count_range(pre_bunch, 20, 25), len(pre_bunch))
rate_post_15_20 = rate_pct(count_range(post_bunch, 15, 20), len(post_bunch))
rate_post_20_25 = rate_pct(count_range(post_bunch, 20, 25), len(post_bunch))

bunching_table = pd.DataFrame({
    "Giai đoạn": ["Pre-policy", "Post-policy"],
    "Mẫu số hợp lệ": [len(pre_bunch), len(post_bunch)],
    "Tỷ lệ [15-20p) (%)": [rate_pre_15_20, rate_post_15_20],
    "Tỷ lệ [20-25p) (%)": [rate_pre_20_25, rate_post_20_25],
})

bunching_table["Bunching_Ratio"] = bunching_table.apply(
    lambda r: r["Tỷ lệ [15-20p) (%)"] / r["Tỷ lệ [20-25p) (%)"] 
    if pd.notna(r["Tỷ lệ [20-25p) (%)"]) and r["Tỷ lệ [20-25p) (%)"] > 0 else np.nan, axis=1
)
bunching_table["Chênh lệch (%)"] = bunching_table["Tỷ lệ [15-20p) (%)"] - bunching_table["Tỷ lệ [20-25p) (%)"]

print("Lưu ý: Tỷ lệ tính trên tổng số phiên idle hợp lệ của từng giai đoạn.")
print(bunching_table.round(2).to_markdown(index=False))
bunching_table.to_csv(f"{OUTPUT_DIR}/3.3.4_bunching.csv", index=False)

# Ép màu và thứ tự chuẩn
period_order = ["Pre", "Post"]
palette_colors = sns.color_palette("muted")
period_palette = {"Pre": palette_colors[0], "Post": palette_colors[1]}

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = df[(df["T_idle_min"] >= 0) & (df["T_idle_min"] <= 60)]

sns.histplot(
    data=plot_df, x="T_idle_min", hue="Policy_Period",
    hue_order=period_order, palette=period_palette,
    bins=60, stat="density", common_norm=False, element="step", ax=ax
)

# Legend thủ công khớp 100% với màu đã ép
legend_elements = [
    mpatches.Patch(color=period_palette["Pre"], alpha=0.5, label='Pre-policy'),
    mpatches.Patch(color=period_palette["Post"], alpha=0.5, label='Post-policy'),
    Line2D([0], [0], color='red', linestyle='--', lw=2, label='Ngưỡng ân hạn 20p')
]

ax.axvline(20, color="red", linestyle="--", linewidth=2)
ax.set_title("Phân phối mật độ T_idle (0-60 phút) - Pre vs Post Policy")
ax.set_xlabel("Thời gian nhàn rỗi T_idle (phút)")
ax.set_ylabel("Mật độ phân phối (Density)")
ax.legend(handles=legend_elements, title="Chú thích", loc='upper right')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/3.3.4_bunching_plot.png", dpi=300)
plt.show()

print(f"\n[HOÀN TẤT] Đã lưu kết quả vào {OUTPUT_DIR}")